In [ ]:
!pip install pertpy

In [ ]:
import pertpy as pt
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
adata = sc.read_h5ad('/workspace/PDAC_Work_Dir/PDAC_Final/Human_Atlas_Harmonised_genes_filtered.h5ad')

In [ ]:
adata.obs.TreatmentType.value_counts()

In [ ]:
# Aggregate endpoints into "Radiotherapy" vs "No Radiotherapy"
# First, map TreatmentType categories to binary radiotherapy status
adata.obs['Radiotherapy_Status'] = adata.obs['TreatmentType'].map(
    lambda x: "Radiotherapy" if "Radiotherapy" in str(x) else "No Radiotherapy"
)

# Show counts for aggregated categories
print(adata.obs['Radiotherapy_Status'].value_counts())

In [ ]:
adata.obs.groupby(['TreatmentType', 'Radiotherapy_Status']).size().unstack()

In [ ]:
endo_types = ["Lymphatic Endothelial Cell", "Tumor-Associated Endothelial Cell", "Vascular Endothelial Cell"]
adata_endo = adata[adata.obs["Level_4"].isin(endo_types)].copy()

count_table_endo = (
    adata_endo.obs.groupby(["Dataset_ID", "Level_4"]).size().unstack(fill_value=0)
)

import anndata

adata_endo_counts = anndata.AnnData(X=count_table_endo.values, 
                                    obs=pd.DataFrame(index=count_table_endo.index),
                                    var=pd.DataFrame(index=count_table_endo.columns))

# Add Radiotherapy_Status to obs by mapping from adata.obs using Dataset_ID
adata_endo_counts.obs["Radiotherapy_Status"] = adata_endo_counts.obs.index.map(
    adata.obs.drop_duplicates("Dataset_ID").set_index("Dataset_ID")["Radiotherapy_Status"]
)

In [ ]:
adata_endo_counts

In [ ]:
adata_endo_counts.obs.index.name = None 

In [ ]:
adata_endo_counts.obs['Dataset_ID'] = adata_endo_counts.obs.index.copy()
adata_endo_counts.obs.head()

In [ ]:
sccoda = pt.tl.Sccoda()
sccoda_data = sccoda.load(
    adata_endo_counts,
    type="sample_level",
    generate_sample_level=False,
    cell_type_identifier="Level_4",
    sample_identifier="Dataset_ID",
    covariate_obs=["Radiotherapy_Status"],
)
sccoda_data["coda"].obs_names = sccoda_data["coda"].obs_names.astype(str)
sccoda_data

In [ ]:
model = sccoda.prepare(
    sccoda_data,
    formula="Radiotherapy_Status",
    reference_cell_type="automatic",
    modality_key="coda"

)
sccoda.run_nuts(
    model,
    modality_key="coda",
)

sccoda.summary(model, modality_key="coda")

In [ ]:
sccoda_results = {}
for ref in endo_types:
    print(f"\n=== Running scCODA for reference cell type: {ref} ===")
    sccoda = pt.tl.Sccoda()
    model = sccoda.prepare(
        sccoda_data,
        formula="Radiotherapy_Status",
        reference_cell_type=ref,
        modality_key="coda"
    )
    sccoda.run_nuts(
        model,
        modality_key="coda",
    )
    # summary = sccoda.summary(model, modality_key="coda")
    sccoda_results[ref] = {
        "model": model,
        # "summary": summary
    }
    print(f"Results for reference cell type: {ref}")
    print(sccoda.summary(model, modality_key="coda"))
    print(f"=== Finished {ref} ===\n")

# use vascular endothelial cells as reference cell type

In [ ]:
sccoda = pt.tl.Sccoda()
sccoda_data = sccoda.load(
    adata_endo_counts,
    type="sample_level",
    generate_sample_level=False,
    cell_type_identifier="Level_4",
    sample_identifier="Dataset_ID",
    covariate_obs=["Radiotherapy_Status"],
)
sccoda_data["coda"].obs_names = sccoda_data["coda"].obs_names.astype(str)
sccoda_data

In [ ]:
model = sccoda.prepare(
    sccoda_data,
    formula="Radiotherapy_Status",
    reference_cell_type="Vascular Endothelial Cell",
    modality_key="coda"

)
sccoda.run_nuts(
    model,
    modality_key="coda",
)

sccoda.summary(model, modality_key="coda")

In [ ]:
print(sccoda_data["coda"].var.index.dtype)
print(sccoda_data["coda"].obs.index.dtype)

In [ ]:
sccoda_data["coda"].var.index = sccoda_data["coda"].var.index.astype(str)
sccoda_data["coda"].obs.index = sccoda_data["coda"].obs.index.astype(str)

In [ ]:
import os
os.chdir('/workspace/PDAC_Work_Dir/PDAC_Final/PDAC_Revisions')

In [ ]:
pwd

In [ ]:
# Level_3 palette dictionary
level_3_palette = {
    "Macrophage":                   "#8b4513",
    "Acinar (REG+) Cell":           "#8a2be2",
    "Malignant Cell - Mesenchymal": "#483d8b",
    "B Cell":                       "#b03060",
    "Cancer Associated Fibroblast": "#dc143c",
    "Malignant Cell - Epithelial":  "#008b8b",
    "Monocyte":                     "#808000",
    "Epsilon Cell":                 "#ff8c00",
    "CD4+ T Cell":                  "#8fbc8f",
    "Ductal Cell":                  "#ffd700",
    "Ductal Cell (atypical)":       "#00ff00",
    "Fibroblast":                   "#dc143c",
    "NK Cell":                      "#ff4500",
    "Mixed T Cell":                 "#800080",
    "Gamma Cell":                   "#add8e6",
    "CD8+ T Cell":                  "#00008b",
    "Neutrophil":                   "#006400",
    "Malignant Cell - EMT":         "#bc8f8f",
    "Endothelial Cell":             "#dda0dd",
    "Alpha Cell":                   "#1e90ff",
    "ADM Cell":                     "#00ff00",
    "Dendritic Cell":               "#00fa9a",
    "Beta Cell":                    "#ff00ff",
    "Acinar Idling Cell":           "#dc143c",
    "Delta Cell":                   "#00bfff",
    "Acinar Cell":                  "#00ffff",
    "Plasma Cell":                  "#0000ff",
    "Adipocyte":                    "#adff2f",
    "Mast Cell":                    "#ffe4c4",
    "Other Endocrine":              "#000000",
    "Pericyte":                     "#ff1493",
    "Schwann Cell":                 "#a5a5a5",
    "Smooth Muscle Cell":           "#b22222",
}

# Level_4 palette dictionary
level_4_palette = {
    'Acinar (REG+) Cell': '#a6cee3',
    'Acinar Cell': '#b2df8a',
    'Acinar Idling Cell': '#fdbf6f',
    'ADM Cell': '#cab2d6',
    'Adipocyte': '#ffff99',
    'Alpha Cell': '#b15928',
    'apCAF': '#1f78b4',
    'B Cell - Activated': '#1f78b4',
    'B Cell - Germinal Center': '#33a02c',
    'B Cell - Memory': '#e31a1c',
    'B Cell - Naive': '#6a3d9a',
    'B-reg': '#b15928',
    'Beta Cell': '#1f78b4',
    'CD4+ Memory T Cell': '#a6cee3',
    'CD4+ Naive T Cell': '#b2df8a',
    'CD4+ Th1 Cell': '#fb9a99',
    'CD4+ Th17 Cell': '#e31a1c',
    'CD4+ Th2 Cell': '#cab2d6',
    'CD4+ Th22 Cell': '#1f78b4',
    'CD8+ Effector T Cell': '#a6cee3',
    'CD8+ Memory T Cell': '#b2df8a',
    'CD8+ Naive T Cell': '#fb9a99',
    'CD8+ Terminal Effector T Cell': '#cab2d6',
    'CD8+ Tissue-Resident Memory T Cell': '#b15928',
    'CD8+ Exhausted T Cell': '#33a02c',
    'Dendritic Cell - Activated': '#e377c2',
    'Dendritic Cell - cDC1': '#1f78b4',
    'Dendritic Cell - cDC2': '#e31a1c',
    'Dendritic Cell - pDC': '#6a3d9a',
    'Delta Cell': '#bcbd22',
    'Double Positive CD4+CD8+ T Cell': '#e31a1c',
    'Ductal Cell': '#17becf',
    'Ductal Cell (atypical)': '#9edae5',
    'Epsilon Cell': '#393b79',
    'Fibroblast': '#5254a3',
    'Gamma Cell': '#6b6ecf',
    'γδ T Cell (Vδ1)': '#6a3d9a',
    'iCAF': '#9c9ede',
    'Lymphatic Endothelial Cell': '#637939',
    'Macrophage - angiogenic TAM': '#1f78b4',
    'Macrophage - antigen processing TAM': '#ff7f00',
    'Macrophage - CD3+ TAM': '#e31a1c',
    'Macrophage - C1Q+ TAM': '#6a3d9a',
    'Macrophage - lipid processing TAM': '#b15928',
    'Malignant Cell - Pit Like': "#a6cee3",
    'Malignant Cell - Acinar-like': "#b2df8a",
    'Malignant Cell - Epithelial': "#fb9a99",
    'Malignant Cell - Hypoxia': "#fdbf6f",
    'Malignant Cell - EMT': "#cab2d6",
    'Malignant Cell - Highly Proliferative': "#e31a1c",
    'Malignant Cell - Mesenchymal': "#fed683",
    'Malignant Cell - Highly Invasive': "#6a3d9a",
    'Malignant Cell - Senescence': "#1f78b4",
    'Malignant Cell - Apoptotic': "#b15928",
    'Mast Cell': '#393b79',
    'Monocyte': '#6b6ecf',
    'myCAF': '#9c9ede',
    'Neutrophil - N0': '#1f78b4',
    'Neutrophil - N1': '#33a02c',
    'Neutrophil - N2': '#b15928',
    'NK Cell': '#cedb9c',
    'Other Endocrine': '#8c6d31',
    'Pericyte': '#bd9e39',
    'Plasma Cell': '#e7ba52',
    'Plasmablast': '#e7969c',
    'Schwann Cell': '#843c39',
    'Smooth Muscle Cell': '#ad494a',
    'T-reg': '#b15928',
    'Tumor-Associated Endothelial Cell': '#7b4173',
    'Vascular Endothelial Cell': '#a55194',
}

In [ ]:
pwd

In [ ]:
sc.set_figure_params(dpi_save=300, dpi=300)
# sccoda.plot_boxplots returns None, so we use plt.gcf() to get the current figure
sccoda.plot_boxplots(
    sccoda_data,
    modality_key="coda",
    feature_name='Radiotherapy_Status',
    add_dots=False,
    dpi=300,
    palette={"Radiotherapy": "#e89071", "No Radiotherapy": "#6dc9b6"}  # Use new palette per legend
)
plt.gcf().savefig("sccoda_boxplot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
pwd

In [ ]:
sccoda.set_fdr(sccoda_data, modality_key="coda", est_fdr=0.4)

In [ ]:
intercept_df_endo = sccoda.get_intercept_df(sccoda_data, modality_key="coda")
intercept_df_endo.to_csv("intercept_df_endothelial.csv")


In [ ]:
intercept_df_endo

In [ ]:
effect_df_endo = sccoda.get_effect_df(sccoda_data)
effect_df_endo.to_csv("effect_df_endothelial.csv")
effect_df_endo

In [ ]:
tme_cell_types = ["CD4+ T Cell", "CD8+ T Cell", "Macrophage", "Monocyte", "Cancer Associated Fibroblast"]
adata_tme = adata[adata.obs["Level_3"].isin(tme_cell_types)].copy()

count_table_tme = (
    adata_tme.obs.groupby(["Dataset_ID", "Level_3"]).size().unstack(fill_value=0)
)

In [ ]:
count_table_tme.head()

In [ ]:
adata_tme_counts = anndata.AnnData(X=count_table_tme.values, 
                                    obs=pd.DataFrame(index=count_table_tme.index),
                                    var=pd.DataFrame(index=count_table_tme.columns))

# Add Radiotherapy_Status to obs by mapping from adata.obs using Dataset_ID
adata_tme_counts.obs["Radiotherapy_Status"] = adata_tme_counts.obs.index.map(
    adata.obs.drop_duplicates("Dataset_ID").set_index("Dataset_ID")["Radiotherapy_Status"]
)

In [ ]:
adata_tme_counts

In [ ]:
adata_tme_counts.obs.index.name = None 

In [ ]:
adata_tme_counts.obs['Dataset_ID'] = adata_tme_counts.obs.index.copy()
adata_tme_counts.obs.head()

In [ ]:
sccoda_tme = pt.tl.Sccoda()
sccoda_data_tme = sccoda_tme.load(
    adata_tme_counts,
    type="sample_level",
    generate_sample_level=False,
    cell_type_identifier="Level_3",
    sample_identifier="Dataset_ID",
    covariate_obs=["Radiotherapy_Status"],
)
sccoda_data_tme["coda"].obs_names = sccoda_data_tme["coda"].obs_names.astype(str)
sccoda_data_tme

In [ ]:
model = sccoda_tme.prepare(
    sccoda_data_tme,
    formula="Radiotherapy_Status",
    reference_cell_type="automatic",
    modality_key="coda",
    automatic_reference_absence_threshold=0.5  # Increase threshold as suggested by error
)
sccoda_tme.run_nuts(
    model,
    modality_key="coda",
)

sccoda_tme.summary(model, modality_key="coda")

In [ ]:
print(sccoda_data_tme["coda"].var.index.dtype)
print(sccoda_data_tme["coda"].obs.index.dtype)

In [ ]:
sccoda_data_tme["coda"].var.index = sccoda_data_tme["coda"].var.index.astype(str)
sccoda_data_tme["coda"].obs.index = sccoda_data_tme["coda"].obs.index.astype(str)

In [ ]:
sc.set_figure_params(dpi_save=300, dpi=300)
palette = {"Radiotherapy": "#e89071", "No Radiotherapy": "#6dc9b6"}  # Use new palette per legend

ax = sccoda_tme.plot_boxplots(
    sccoda_data_tme, 
    modality_key="coda", 
    feature_name='Radiotherapy_Status',
    add_dots=False, 
    dpi=300, 
    palette=palette
)

# Save the actual figure referenced by `ax`
fig = ax.get_figure() if hasattr(ax, "get_figure") else plt.gcf()
fig.savefig("sccoda_boxplot_TME.png", dpi=300, bbox_inches="tight")
plt.close(fig)  # Close so it does not display a duplicate in Jupyter

In [ ]:
sccoda_tme.set_fdr(sccoda_data_tme, modality_key="coda", est_fdr=0.4)

In [ ]:
intercept_df_tme = sccoda_tme.get_intercept_df(sccoda_data_tme, modality_key="coda")
intercept_df_tme.to_csv("intercept_df_tme.csv")

intercept_df_tme
effect_df_tme = sccoda_tme.get_effect_df(sccoda_data_tme)
effect_df_tme.to_csv("effect_df_tme.csv")
effect_df_tme

In [ ]:
intercept_df_tme